In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException
import time
import pandas as pd

jobssearch = ['Python Developer', 'Data Engineer', 'AI Developer', 'Data Scientist']

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

list_jobs = []
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15, ignored_exceptions=[StaleElementReferenceException])  # 

for job_title_query in jobssearch:
    driver.get('https://wuzzuf.net/jobs/egypt')

  
    time.sleep(3)

   
    for attempt in range(3):
        try:
            search_box = wait.until(EC.element_to_be_clickable((By.NAME, 'q')))
            search_box.clear()
            search_box.send_keys(job_title_query)
            break  # success, exit retry loop
        except StaleElementReferenceException:
            print(f"Stale on search box, retrying... ({attempt+1})")
            time.sleep(2)

   
    for attempt in range(3):
        try:
            search_btn = wait.until(EC.element_to_be_clickable((
                By.XPATH, '//*[@id="app"]/div/div/main/div[2]/div[1]/form/button'
            )))
            driver.execute_script("arguments[0].click();", search_btn)
            break
        except StaleElementReferenceException:
            print(f"Stale on search button, retrying... ({attempt+1})")
            time.sleep(2)

    for page in range(3):
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, 'css-pkv5jc')))
        time.sleep(2)

        jobs = driver.find_elements(By.CLASS_NAME, 'css-pkv5jc')

        for job in jobs:
            try:
                title = job.find_element(By.CLASS_NAME, 'css-o171kl').text
                company = job.find_element(By.CLASS_NAME, 'css-ipsyv7').text
                location = job.find_element(By.CLASS_NAME, 'css-16x61xq').text
                tags = job.find_element(By.CLASS_NAME, 'css-5jhz9n').text.split('\n')
                try:
                    exp = job.find_element(
                        By.CSS_SELECTOR, 'div.css-1rhj4yg > div:nth-child(2) > span'
                    ).text
                except:
                    exp = 'No Exp'

                list_jobs.append({
                    'job_title': title,
                    'company': company,
                    'location': location,
                    'tags': tags,
                    'exp': exp,
                    'search_query': job_title_query
                })
            except StaleElementReferenceException:
                print("Stale job card, skipping...")
                continue
            except Exception as e:
                print(f"Skipping card: {e}")
                continue

        if page < 2:
            try:
                next_btn = wait.until(EC.element_to_be_clickable((
                    By.XPATH, '//*[@id="app"]/div/div[3]/div/div/div[3]/ul/li[7]/button'
                )))
                driver.execute_script("arguments[0].click();", next_btn)
            except Exception as e:
                print(f"Could not go to next page: {e}")
                break

driver.quit()
df = pd.DataFrame(list_jobs)
df.to_csv('Jobs.csv', index=False)
print(f"Saved {len(df)} jobs to Jobs.csv")